In [0]:
import pandas as pd

df = pd.read_csv("/Volumes/main/default/raw_data/SentinelMesh_Final_Subset.csv")
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} colums")
print(df["Label"].value_counts())

Loaded 240,000 rows, 41 colums
Label
BENIGN                     120000
DDOS-ICMP_FLOOD             18646
DDOS-UDP_FLOOD              14188
DDOS-TCP_FLOOD              11666
DDOS-PSHACK_FLOOD           10692
DDOS-RSTFINFLOOD            10682
DDOS-SYN_FLOOD              10660
DDOS-SYNONYMOUSIP_FLOOD      9546
DOS-UDP_FLOOD                8612
DOS-TCP_FLOOD                6999
DOS-SYN_FLOOD                5225
MIRAI-GREETH_FLOOD           2693
MIRAI-UDPPLAIN               2293
MIRAI-GREIP_FLOOD            2049
DDOS-ICMP_FRAGMENTATION      1123
VULNERABILITYSCAN             940
MITM-ARPSPOOFING              793
DDOS-ACK_FRAGMENTATION        733
DDOS-UDP_FRAGMENTATION        719
DNS_SPOOFING                  467
RECON-HOSTDISCOVERY           358
RECON-OSSCAN                  261
RECON-PORTSCAN                216
DOS-HTTP_FLOOD                202
DDOS-HTTP_FLOOD                65
DDOS-SLOWLORIS                 61
DICTIONARYBRUTEFORCE           37
SQLINJECTION                   17
BROWSERHIJA

In [0]:
df.columns = df.columns.str.replace(' ', '_')
spark.createDataFrame(df).write.format("delta").mode("overwrite").saveAsTable("main.default.cic_alerts_raw")
display(spark.table("main.default.cic_alerts_raw").limit(5))

Header_Length,Protocol_Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,cwr_flag_number,ack_count,syn_count,fin_count,rst_count,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ARP,ICMP,IGMP,IPv,LLC,Tot_sum,Min,Max,AVG,Std,Tot_size,IAT,Number,Variance,Label,is_attack
0.32,1,61.93,1882.2465052617408,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.04,0.0,0.03,0.93,0.0,0.97,0.97,86816.0,60.0,1514.0,868.16,604.1431819051044,868.16,5.342698097229E-4,100.0,364988.9842424241,DDOS-ICMP_FRAGMENTATION,1
20.0,6,64.0,54920.83278774388,0.0,0.0,0.0,1.0,1.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,6000.0,60.0,60.0,60.0,0.0,60.0,1.8219947814941407E-5,100.0,0.0,DDOS-PSHACK_FLOOD,1
10.52,6,66.51,873.5676424338988,0.0,0.02,0.0,0.0,0.5,0.0,0.0,50.0,2.0,0.0,0.0,0.01,0.49,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.04,0.0,0.0,0.96,0.96,90248.0,60.0,1514.0,902.48,594.5021596073285,902.48,0.0011447405815124,100.0,353432.8177777776,DDOS-ACK_FRAGMENTATION,1
27.2,6,219.1,135.202870193377,0.0,0.0,0.0,0.1,0.8,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.8,0.2,0.0,0.0,0.0,0.0,1.0,1.0,1046.0,66.0,214.0,104.6,64.12522471262338,104.6,0.0083137989044189,10.0,4112.044444444444,BENIGN,0
32.0,6,232.0,912.7377973146477,0.0,0.0,0.0,0.0,1.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,660.0,66.0,66.0,66.0,0.0,66.0,0.0033185958862304,10.0,0.0,BENIGN,0


In [0]:
## Step 2: Build a Diverse, Stratified Sample of Alerts

# Pull a handful of rows from every attack category (including BENIGN) so
# our agent gets tested against the full variety of attacks, not just the
# most common ones. Capped at 5 per category so common attacks like
# DDOS-ICMP_FLOOD don't drown out rare ones like BACKDOOR_MALWARE.
n_per_category = 5

# group by Label, then take up to n_per_category rows from each group
# (fewer if a category has less than that many rows available)
sample_df = (
    df.groupby('Label', group_keys=False)
    .apply(lambda group: group.sample(n=min(n_per_category, len(group)), random_state=42))
    .reset_index(drop=True)
)

print(f"Sample contains {len(sample_df)} alerts across {sample_df['Label'].nunique()} categories")
print(sample_df['Label'].value_counts())

Sample contains 168 alerts across 34 categories
Label
DNS_SPOOFING               5
BENIGN                     5
VULNERABILITYSCAN          5
SQLINJECTION               5
RECON-PORTSCAN             5
RECON-PINGSWEEP            5
RECON-OSSCAN               5
RECON-HOSTDISCOVERY        5
MITM-ARPSPOOFING           5
MIRAI-UDPPLAIN             5
MIRAI-GREIP_FLOOD          5
MIRAI-GREETH_FLOOD         5
DOS-UDP_FLOOD              5
DOS-TCP_FLOOD              5
DOS-SYN_FLOOD              5
DOS-HTTP_FLOOD             5
XSS                        5
DICTIONARYBRUTEFORCE       5
DDOS-UDP_FRAGMENTATION     5
BROWSERHIJACKING           5
COMMANDINJECTION           5
DDOS-ACK_FRAGMENTATION     5
DDOS-HTTP_FLOOD            5
DDOS-ICMP_FLOOD            5
DDOS-ICMP_FRAGMENTATION    5
DDOS-PSHACK_FLOOD          5
DDOS-RSTFINFLOOD           5
DDOS-SLOWLORIS             5
DDOS-SYNONYMOUSIP_FLOOD    5
DDOS-SYN_FLOOD             5
DDOS-TCP_FLOOD             5
DDOS-UDP_FLOOD             5
UPLOADING_ATTACK  

/home/spark-b90c8ed7-0699-4ffe-91f9-d6/.ipykernel/10014/command-6717476090483146-2952614061:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(n=min(n_per_category, len(group)), random_state=42))


In [0]:
## Step 2: Build a Diverse, Stratified Sample of Alerts

# Pull a handful of rows from every attack category (including BENIGN) so
# our agent gets tested against the full variety of attacks, not just the
# most common ones. Capped at 5 per category so common attacks like
# DDOS-ICMP_FLOOD don't drown out rare ones like BACKDOOR_MALWARE.
n_per_category = 5

# group by Label, then take up to n_per_category rows from each group
# (fewer if a category has less than that many rows available)
sample_df = (
    df.groupby('Label', group_keys=False)
    .apply(lambda group: group.sample(n=min(n_per_category, len(group)), random_state=42))
    .reset_index(drop=True)
)

print(f"Sample contains {len(sample_df)} alerts across {sample_df['Label'].nunique()} categories")
print(sample_df['Label'].value_counts())

Sample contains 168 alerts across 34 categories
Label
DNS_SPOOFING               5
BENIGN                     5
VULNERABILITYSCAN          5
SQLINJECTION               5
RECON-PORTSCAN             5
RECON-PINGSWEEP            5
RECON-OSSCAN               5
RECON-HOSTDISCOVERY        5
MITM-ARPSPOOFING           5
MIRAI-UDPPLAIN             5
MIRAI-GREIP_FLOOD          5
MIRAI-GREETH_FLOOD         5
DOS-UDP_FLOOD              5
DOS-TCP_FLOOD              5
DOS-SYN_FLOOD              5
DOS-HTTP_FLOOD             5
XSS                        5
DICTIONARYBRUTEFORCE       5
DDOS-UDP_FRAGMENTATION     5
BROWSERHIJACKING           5
COMMANDINJECTION           5
DDOS-ACK_FRAGMENTATION     5
DDOS-HTTP_FLOOD            5
DDOS-ICMP_FLOOD            5
DDOS-ICMP_FRAGMENTATION    5
DDOS-PSHACK_FLOOD          5
DDOS-RSTFINFLOOD           5
DDOS-SLOWLORIS             5
DDOS-SYNONYMOUSIP_FLOOD    5
DDOS-SYN_FLOOD             5
DDOS-TCP_FLOOD             5
DDOS-UDP_FLOOD             5
UPLOADING_ATTACK  

/home/spark-b90c8ed7-0699-4ffe-91f9-d6/.ipykernel/10014/command-5070798811226816-2952614061:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(n=min(n_per_category, len(group)), random_state=42))


## Converting Flow Records into Alert Text

The first version of the alert renderer compressed each network flow into a short sentence. That made the agent accurate on obvious flood patterns but weak on lower signal classes such as Mirai, reconnaissance, spoofing, brute force, web attacks, malware, and benign traffic. This revised renderer exposes more observable flow evidence, including protocol mix, packet rate, packet count, packet size statistics, inter arrival time, TCP flag activity, and counter fields.

The ground truth label is intentionally excluded from `alert_text`. The agent sees only the rendered alert text during triage and evaluation.

In [0]:
## Step 3: Convert Raw Rows into Richer Plain English Alerts

def render_alert(row):
    cols = set(row.index)

    def has(c):
        return c in cols

    def num(c, default=0.0):
        try:
            if has(c) and pd.notna(row[c]):
                return float(row[c])
        except Exception:
            pass
        return default

    protocol_cols = [
        "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
        "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"
    ]

    present_proto = [c for c in protocol_cols if has(c)]

    if present_proto:
        vals = {c: num(c) for c in present_proto}
        dom = max(vals, key=vals.get)
        proto_desc = f"{dom} dominant traffic ({vals[dom] * 100:.0f}% of flow)"
    else:
        proto_desc = "unknown protocol mix"

    avg = num("AVG")
    std = num("Std")
    variability = (std / avg) if avg > 0 else 0

    if variability < 0.05:
        size_desc = "highly uniform packet sizes"
    elif variability < 0.20:
        size_desc = "mostly uniform packet sizes"
    else:
        size_desc = "varied packet sizes"

    flag_map = {
        "SYN": "syn_flag_number",
        "ACK": "ack_flag_number",
        "FIN": "fin_flag_number",
        "RST": "rst_flag_number",
        "PSH": "psh_flag_number",
        "ECE": "ece_flag_number",
        "CWR": "cwr_flag_number"
    }

    active_flags = [
        f"{name}={num(col):.2f}"
        for name, col in flag_map.items()
        if has(col) and num(col) > 0
    ]

    flag_desc = (
        "TCP flag activity: " + ", ".join(active_flags)
        if active_flags
        else "TCP flag activity: no TCP flags observed"
    )

    measures = []

    def add(label, col, fmt):
        if has(col):
            measures.append(f"{label}: {fmt.format(num(col))}")

    add("packets per second", "Rate", "{:,.0f}")
    add("packet count", "Number", "{:,.0f}")
    add("average packet length", "AVG", "{:.0f} bytes")
    add("packet length standard deviation", "Std", "{:.0f}")
    add("minimum packet length", "Min", "{:.0f}")
    add("maximum packet length", "Max", "{:.0f}")
    add("total packet size", "Tot_size", "{:.0f}")
    add("total packet size sum", "Tot_sum", "{:.0f}")
    add("inter arrival time", "IAT", "{:.6f} seconds")
    add("SYN count", "syn_count", "{:.0f}")
    add("ACK count", "ack_count", "{:.0f}")
    add("RST count", "rst_count", "{:.0f}")
    add("FIN count", "fin_count", "{:.0f}")
    add("header length", "Header_Length", "{:.0f}")
    add("time to live", "Time_To_Live", "{:.0f}")
    add("variance", "Variance", "{:.2f}")

    pieces = [proto_desc] + measures + [size_desc, flag_desc]
    body = ", ".join(pieces)

    return (
        f"Network flow alert: {body}. "
        f"The true label is not shown to the agent during evaluation."
    )

sample_df["alert_text"] = sample_df.apply(render_alert, axis=1)

for i in range(3):
    print(f"ALERT TEXT: {sample_df.loc[i, 'alert_text']}")
    print(f"TRUE LABEL hidden from agent: {sample_df.loc[i, 'Label']}")
    print()

ALERT TEXT: Network flow alert: IPv dominant traffic (100% of flow), packets per second: 21, packet count: 10, average packet length: 162 bytes, packet length standard deviation: 71, minimum packet length: 66, maximum packet length: 256, total packet size: 162, total packet size sum: 1623, inter arrival time: 0.055822 seconds, SYN count: 0, ACK count: 3, RST count: 0, FIN count: 0, header length: 15, time to live: 158, variance: 5042.23, varied packet sizes, TCP flag activity: ACK=0.30, PSH=0.20. The true label is not shown to the agent during evaluation.
TRUE LABEL hidden from agent: BACKDOOR_MALWARE

ALERT TEXT: Network flow alert: IPv dominant traffic (100% of flow), packets per second: 34, packet count: 10, average packet length: 140 bytes, packet length standard deviation: 70, minimum packet length: 60, maximum packet length: 218, total packet size: 140, total packet size sum: 1400, inter arrival time: 0.030426 seconds, SYN count: 0, ACK count: 4, RST count: 0, FIN count: 0, heade

In [0]:
## Step 4: Save the Sample as a Reusable Table

spark.createDataFrame(sample_df).write.format("delta").mode("overwrite") \
    .saveAsTable("main.default.cic_alerts_sample")

display(spark.table("main.default.cic_alerts_sample").limit(5))

Header_Length,Protocol_Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,cwr_flag_number,ack_count,syn_count,fin_count,rst_count,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ARP,ICMP,IGMP,IPv,LLC,Tot_sum,Min,Max,AVG,Std,Tot_size,IAT,Number,Variance,Label,is_attack,alert_text
15.2,17,157.7,21.074860578848384,0.0,0.0,0.0,0.2,0.3,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.7,0.0,0.0,0.0,0.0,1.0,1.0,1623.0,66.0,256.0,162.3,71.00868491482808,162.3,0.0558224916458129,10.0,5042.233333333334,BACKDOOR_MALWARE,1,"Network flow alert: IPv dominant traffic (100% of flow), packets per second: 21, packet count: 10, average packet length: 162 bytes, packet length standard deviation: 71, minimum packet length: 66, maximum packet length: 256, total packet size: 162, total packet size sum: 1623, inter arrival time: 0.055822 seconds, SYN count: 0, ACK count: 3, RST count: 0, FIN count: 0, header length: 15, time to live: 158, variance: 5042.23, varied packet sizes, TCP flag activity: ACK=0.30, PSH=0.20. The true label is not shown to the agent during evaluation."
17.6,17,99.3,33.811534911466794,0.0,0.0,0.0,0.2,0.4,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,0.0,0.0,0.0,0.0,1.0,1.0,1400.0,60.0,218.0,140.0,69.50619476788462,140.0,0.0304259061813354,10.0,4831.111111111111,BACKDOOR_MALWARE,1,"Network flow alert: IPv dominant traffic (100% of flow), packets per second: 34, packet count: 10, average packet length: 140 bytes, packet length standard deviation: 70, minimum packet length: 60, maximum packet length: 218, total packet size: 140, total packet size sum: 1400, inter arrival time: 0.030426 seconds, SYN count: 0, ACK count: 4, RST count: 0, FIN count: 0, header length: 18, time to live: 99, variance: 4831.11, varied packet sizes, TCP flag activity: ACK=0.40, PSH=0.20. The true label is not shown to the agent during evaluation."
23.5,6,103.75,27.576507097444395,0.125,0.0,0.0,0.125,0.75,0.0,0.0,6.0,0.0,1.0,0.0,0.0,0.625,0.0,0.0,0.0,0.0,0.0,0.75,0.125,0.0,0.125,0.0,0.0,0.875,0.875,695.0,60.0,214.0,86.875,52.71334203568797,86.875,0.0391952395439147,8.0,2778.6964285714284,BACKDOOR_MALWARE,1,"Network flow alert: IPv dominant traffic (88% of flow), packets per second: 28, packet count: 8, average packet length: 87 bytes, packet length standard deviation: 53, minimum packet length: 60, maximum packet length: 214, total packet size: 87, total packet size sum: 695, inter arrival time: 0.039195 seconds, SYN count: 0, ACK count: 6, RST count: 0, FIN count: 1, header length: 24, time to live: 104, variance: 2778.70, varied packet sizes, TCP flag activity: ACK=0.75, FIN=0.12, PSH=0.12. The true label is not shown to the agent during evaluation."
9.6,17,134.1,57.87066829292522,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.1,0.0,0.3,0.0,0.0,0.0,0.0,0.1,0.7,0.0,0.0,0.2,0.0,1.0,1.0,1789.0,74.0,230.0,178.9,65.64627432941899,178.9,0.0177901029586792,10.0,4309.433333333333,BACKDOOR_MALWARE,1,"Network flow alert: IPv dominant traffic (100% of flow), packets per second: 58, packet count: 10, average packet length: 179 bytes, packet length standard deviation: 66, minimum packet length: 74, maximum packet length: 230, total packet size: 179, total packet size sum: 1789, inter arrival time: 0.017790 seconds, SYN count: 1, ACK count: 0, RST count: 0, FIN count: 0, header length: 10, time to live: 134, variance: 4309.43, varied packet sizes, TCP flag activity: SYN=0.10. The true label is not shown to the agent during evaluation."
32.0,6,214.8,866.1801210168721,0.0,0.0,0.0,0.0,1.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,660.0,66.0,66.0,66.0,0.0,66.0,0.0035226106643676,10.0,0.0,BENIGN,0,"Network flow alert: HTTPS dominant traffic (100% of flow), packets per second: 866, packet count: 10, average packet length: 66 bytes, packet length standard deviation: 0, minimum packet length: 66, maximum packet length: 66, total packet size: 66,